# Feature Engineering

Build the modeling table from `health_and_wellness_no_outliers.csv`.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "dataset" / "health_and_wellness_no_outliers.csv"
OUTPUT_PATH = PROJECT_ROOT / "dataset" / "health_and_wellness_feature_engineered.csv"

raw_df = pd.read_csv(DATA_PATH)
raw_df.head()

,Id,Section,Name,Price,Total Sold,Total Reviews,Shop Location
0,4799179886,Acne Care,DHC Vitamin B-Mix วิตามินบีรวม (สำหรับ 20 วัน),75.0,NaN,NaN,Pathum Thani
1,4795171651,Acne Care,ซิงค์ Vistra Zinc วิสทร้า ซิงค์ 15 มก. ขนาด 20...,79.0,NaN,NaN,Chiang Mai
2,4790242641,Acne Care,Blackmores แบลคมอร์ส Bio Zinc A Chelate (90 T...,224.0,NaN,NaN,Surin
3,4789940606,Acne Care,1แถม1 กลูต้าวิตมี กลูต้าส้มเลือด Gluta With Me...,290.0,NaN,NaN,Udon Thani
4,4787161067,Acne Care,🎌 DHC Vitamin B-Mix Persistent วิตามินบีรวม แบ...,149.0,NaN,NaN,Bangkok


## Column Explanations

| Column | Explanation |
|---|---|
| `target_total_sold` | The prediction target. This comes from the original `Total Sold` column and represents how many units were sold. Missing values are filled with `0`. |
| `log_price_thb` | Log-transformed product price in Thai baht, calculated as `log(1 + Price)`. This reduces the effect of very expensive products. |
| `log_total_reviews` | Log-transformed review count, calculated as `log(1 + Total Reviews)`. This keeps products with very high review counts from dominating the model. |
| `price_vs_section_mean` | The product price minus the average price of products in the same section. Positive values mean the product is more expensive than its section average; negative values mean cheaper. |
| `price_ratio_to_section_mean` | The product price divided by the average price of its section. A value of `1.00` means equal to the section average, `1.20` means 20% above average, and `0.80` means 20% below average. |
| `price_rank_in_section` | The percentile rank of the product price within its section. Values are between `0` and `1`; higher values mean the product is relatively more expensive compared with products in the same section. |
| `has_reviews` | Binary indicator for whether the product has reviews. `1` means `Total Reviews > 0`; `0` means no reviews. |
| `name_length` | Number of characters in the product name. This can capture how detailed or keyword-heavy the product title is. |
| `name_word_count` | Number of words in the product name. This is another measure of product-title detail. |
| `section_*` | One-hot encoded section/category columns. Each product gets `1` for its own section and `0` for all other sections. Example: `section_Acne_Care`. |
| `shop_location_*` | One-hot encoded shop-location columns. Each product gets `1` for its shop location and `0` for all other locations. Example: `shop_location_Bangkok`. |

The one-hot encoded columns are created for every category that exists in the raw dataset, so the final table includes all product sections and all shop locations.

## Build Features

In [3]:
def clean_category_value(value):
    """Create readable, stable dummy-column suffixes from category values."""
    value = "Unknown" if pd.isna(value) else str(value).strip()
    value = re.sub(r"\W+", "_", value, flags=re.UNICODE).strip("_")
    return value or "Unknown"


df = raw_df.copy()

df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Total Sold"] = pd.to_numeric(df["Total Sold"], errors="coerce").fillna(0)
df["Total Reviews"] = pd.to_numeric(df["Total Reviews"], errors="coerce").fillna(0)
df["Section"] = df["Section"].fillna("Unknown")
df["Shop Location"] = df["Shop Location"].fillna("Unknown")
df["Name"] = df["Name"].fillna("")

section_mean_price = df.groupby("Section")["Price"].transform("mean")

features = pd.DataFrame(index=df.index)
features["target_total_sold"] = df["Total Sold"]
features["log_price_thb"] = np.log1p(df["Price"])
features["log_total_reviews"] = np.log1p(df["Total Reviews"])
features["price_vs_section_mean"] = df["Price"] - section_mean_price
features["price_ratio_to_section_mean"] = df["Price"] / section_mean_price.replace(0, np.nan)
features["price_rank_in_section"] = df.groupby("Section")["Price"].rank(method="average", pct=True)
features["has_reviews"] = (df["Total Reviews"] > 0).astype(int)
features["name_length"] = df["Name"].str.len()
features["name_word_count"] = df["Name"].str.split().str.len().fillna(0).astype(int)

categorical_df = pd.DataFrame(
    {
        "section": df["Section"].map(clean_category_value),
        "shop_location": df["Shop Location"].map(clean_category_value),
    }
)

dummy_features = pd.get_dummies(
    categorical_df,
    columns=["section", "shop_location"],
    prefix=["section", "shop_location"],
    dtype=int,
)

feature_df = pd.concat([features, dummy_features], axis=1)
feature_df = feature_df.replace([np.inf, -np.inf], np.nan)
feature_df = feature_df.fillna(0)

feature_df.head()

,target_total_sold,log_price_thb,log_total_reviews,price_vs_section_mean,price_ratio_to_section_mean,price_rank_in_section,has_reviews,name_length,name_word_count,section_Acne_Care,...,shop_location_Trang,shop_location_Trat,shop_location_Ubon_Ratchathani,shop_location_Udon_Thani,shop_location_Unknown,shop_location_Uthai_Thani,shop_location_Uttaradit,shop_location_Yala,shop_location_Yasothon,shop_location_ต_างประเทศ
0,0.0,4.330733,0.0,-350.756173,0.176157,0.024691,0,46,7,1,...,0,0,0,0,0,0,0,0,0,0
1,0.0,4.382027,0.0,-346.756173,0.185552,0.043210,0,53,10,1,...,0,0,0,0,0,0,0,0,0,0
2,0.0,5.416100,0.0,-201.756173,0.526123,0.345679,0,118,18,1,...,0,0,0,0,0,0,0,0,0,0
3,0.0,5.673323,0.0,-135.756173,0.681141,0.537037,0,72,8,1,...,0,0,0,1,0,0,0,0,0,0
4,0.0,5.010635,0.0,-276.756173,0.349966,0.160494,0,119,12,1,...,0,0,0,0,0,0,0,0,0,0


## Save Final Table

In [4]:
feature_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved feature table to: {OUTPUT_PATH}")
print(f"Rows: {feature_df.shape[0]:,}")
print(f"Columns: {feature_df.shape[1]:,}")

Saved feature table to: /Users/tanatnitchunounsri/Desktop/hs-practical-machine-learning-final-project/dataset/health_and_wellness_feature_engineered.csv
Rows: 2,610
Columns: 111


In [5]:
feature_df.filter(
    regex=r"^(target_total_sold|log_price_thb|log_total_reviews|price_|has_reviews|name_|section_|shop_location_)"
).head()

,target_total_sold,log_price_thb,log_total_reviews,price_vs_section_mean,price_ratio_to_section_mean,price_rank_in_section,has_reviews,name_length,name_word_count,section_Acne_Care,...,shop_location_Trang,shop_location_Trat,shop_location_Ubon_Ratchathani,shop_location_Udon_Thani,shop_location_Unknown,shop_location_Uthai_Thani,shop_location_Uttaradit,shop_location_Yala,shop_location_Yasothon,shop_location_ต_างประเทศ
0,0.0,4.330733,0.0,-350.756173,0.176157,0.024691,0,46,7,1,...,0,0,0,0,0,0,0,0,0,0
1,0.0,4.382027,0.0,-346.756173,0.185552,0.043210,0,53,10,1,...,0,0,0,0,0,0,0,0,0,0
2,0.0,5.416100,0.0,-201.756173,0.526123,0.345679,0,118,18,1,...,0,0,0,0,0,0,0,0,0,0
3,0.0,5.673323,0.0,-135.756173,0.681141,0.537037,0,72,8,1,...,0,0,0,1,0,0,0,0,0,0
4,0.0,5.010635,0.0,-276.756173,0.349966,0.160494,0,119,12,1,...,0,0,0,0,0,0,0,0,0,0
